# Deep Cosmos — 4-Scene Animated Persian Science Documentary (HD MP4 + Neural Voice)
Tap **Runtime → Run all** to render the 4-scene animated physics documentary (Orbital Hook, Prism Spectrum, Rayleigh Molecular Scattering, Sunset Horizon Geometry), preview it right inside Colab, and automatically download both `deep-cosmos-documentary.mp4` and `deep-cosmos-youtube-package.zip`.


In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg espeak-ng fonts-noto-core fonts-dejavu-core
!pip -q install pillow numpy arabic-reshaper python-bidi edge-tts


In [ ]:
import asyncio, shutil, subprocess, zipfile
from pathlib import Path
if Path('/content/YouTube-bot').exists(): shutil.rmtree('/content/YouTube-bot')
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/studio1111/YouTube-bot.git', '/content/YouTube-bot'], check=True)
subprocess.run(['python3', '/content/YouTube-bot/src/yt_agent/pro_documentary.py'], check=True)
work = Path('/tmp/pro_yt_episode')
script_lines = (work / 'script-fa.txt').read_text(encoding='utf-8').strip().splitlines()
try:
    import edge_tts
    async def _voice():
        c = edge_tts.Communicate(' ... '.join(script_lines), 'fa-IR-FaridNeural', rate='-5%')
        await c.save(str(work / 'neural_fa.mp3'))
    asyncio.run(_voice())
    if (work / 'neural_fa.mp3').exists() and (work / 'neural_fa.mp3').stat().st_size > 1000:
        subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(work/'neural_fa.mp3'),'-i',str(work/'ambient_bed.wav'),'-filter_complex','[0:a]volume=1.4,apad=whole_dur=24[v];[1:a]volume=0.28[b];[v][b]amix=inputs=2:duration=shortest','-t','24',str(work/'narration.wav')], check=True)
        subprocess.run(['ffmpeg','-y','-loglevel','error','-framerate','20','-i',str(work/'frames/frame-%04d.jpg'),'-i',str(work/'narration.wav'),'-c:v','libx264','-preset','medium','-crf','22','-c:a','aac','-b:a','160k','-shortest','-pix_fmt','yuv420p',str(work/'deep-cosmos-documentary.mp4')], check=True)
        with zipfile.ZipFile(work/'deep-cosmos-youtube-package.zip','w',compression=zipfile.ZIP_DEFLATED) as zf:
            for fn in ['deep-cosmos-documentary.mp4','thumbnail.jpg','narration.wav','script-fa.txt','captions-fa.srt','manifest.json']:
                zf.write(work/fn, arcname=fn)
except Exception as e:
    print('Fallback voice active:', e)
from IPython.display import Video, display
display(Video(str(work/'deep-cosmos-documentary.mp4'), embed=True, width=720))
from google.colab import files
files.download(str(work/'deep-cosmos-documentary.mp4'))
files.download(str(work/'deep-cosmos-youtube-package.zip'))
